In [ ]:
# ADLS Connection Configuration
storage_account = 
client_id       = 
tenant_id       = 
client_secret   = 

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("ADLS Connected")

In [ ]:
from pyspark.sql.functions import col

storage_account  = "adlsadylearning"
container        = "ady-adf-practice"
silver_path      = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Silver"
gold_path        = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Gold"

# Step 1 — Check Silver counts
df_order_details = spark.read.format("delta").load(f"{silver_path}/order_details")
df_orders        = spark.read.format("delta").load(f"{silver_path}/orders")
print("order_details:", df_order_details.count())
print("orders:", df_orders.count())

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
import pandas as pd

# ── CONFIG ──────────────────────────────────────────
storage_account = "adlsadylearning"
container       = "ady-adf-practice"
silver_path     = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Silver"
gold_path       = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Gold"

# ── GENERATE DATE RANGE ─────────────────────────────
# Covering all Northwind orders (2013 - 2015) + buffer
date_range = pd.date_range(start="2010-01-01", end="2020-12-31", freq="D")
df_date    = spark.createDataFrame(pd.DataFrame({"full_date": date_range}))

# ── TRANSFORMATIONS ─────────────────────────────────
df_date = df_date \
    .withColumn("date_key",     date_format(col("full_date"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("day",          dayofmonth(col("full_date"))) \
    .withColumn("month",        month(col("full_date"))) \
    .withColumn("month_name",   date_format(col("full_date"), "MMMM")) \
    .withColumn("quarter",      quarter(col("full_date"))) \
    .withColumn("quarter_name", concat(lit("Q"), quarter(col("full_date")).cast("string"))) \
    .withColumn("year",         year(col("full_date"))) \
    .withColumn("week_of_year", weekofyear(col("full_date"))) \
    .withColumn("day_of_week",  date_format(col("full_date"), "EEEE")) \
    .withColumn("is_weekend",   when(dayofweek(col("full_date")).isin([1, 7]), True).otherwise(False)) \
    .withColumn("is_weekday",   when(dayofweek(col("full_date")).isin([1, 7]), False).otherwise(True))

# ── REORDER COLUMNS ─────────────────────────────────
df_date = df_date.select(
    "date_key", "full_date", "day", "month", "month_name",
    "quarter", "quarter_name", "year", "week_of_year",
    "day_of_week", "is_weekend", "is_weekday"
)

# ── AUDIT COLUMNS ───────────────────────────────────
df_date = df_date \
    .withColumn("created_at", current_timestamp()) \
    .withColumn("updated_at", current_timestamp())

# ── VALIDATE ────────────────────────────────────────
print("Row count:", df_date.count())
df_date.show(5)

# ── WRITE GOLD ──────────────────────────────────────
df_date.write.format("delta") \
             .mode("overwrite") \
             .save(f"{gold_path}/dim_date")

print("gold.dim_date written successfully")

In [ ]:
from delta.tables import DeltaTable

# ── READ SILVER ─────────────────────────────────────
df_categories = spark.read.format("delta").load(f"{silver_path}/categories")

# ── ADD SURROGATE KEY ───────────────────────────────
# category_key = category_id (small table, natural key works fine)
df_categories = df_categories \
    .withColumn("category_key", 
                col("category_id"))

# ── SELECT FINAL COLUMNS ────────────────────────────
df_categories = df_categories.select(
    "category_key",
    "category_id",
    "category_name",
    "description",
    "created_at",
    "updated_at"
)

# ── FIRST TIME LOAD CHECK ───────────────────────────
def table_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except:
        return False

if table_exists(f"{gold_path}/dim_categories"):
    print("Existing table — running MERGE (SCD Type 1)")
    # delta_table using gold file as it exist
    delta_table = DeltaTable.forPath(spark, f"{gold_path}/dim_categories")
    # MERGE. Taking the gold delta and silver delta as inputs
    delta_table.alias("target").merge(
        df_categories.alias("source"),
        "target.category_key = source.category_key"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    print("First time load — writing fresh")
    df_categories.write.format("delta") \
                       .mode("overwrite") \
                       .save(f"{gold_path}/dim_categories")
print("gold.dim_categories written successfully")


In [ ]:
# ── READ SILVER ─────────────────────────────────────
df_shippers = spark.read.format("delta").load(f"{silver_path}/shippers")

# ── ADD SURROGATE KEY ───────────────────────────────
df_shippers = df_shippers \
    .withColumn("shipper_key", col("shipper_id"))

# ── SELECT FINAL COLUMNS ────────────────────────────
df_shippers = df_shippers.select(
    "shipper_key",
    "shipper_id",
    "company_name",
    "created_at",
    "updated_at"
)

# ── FIRST TIME LOAD CHECK ───────────────────────────
def table_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except:
        return False

if table_exists(f"{gold_path}/dim_shippers"):
    print("Existing table — running MERGE (SCD Type 1)")
    delta_table = DeltaTable.forPath(spark, f"{gold_path}/dim_shippers")
    delta_table.alias("target").merge(
        df_shippers.alias("source"),
        "target.shipper_key = source.shipper_key"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    print("First time load — writing fresh")
    df_shippers.write.format("delta") \
                     .mode("overwrite") \
                     .save(f"{gold_path}/dim_shippers")

print("gold.dim_shippers written successfully")


In [ ]:
# ── READ SILVER ─────────────────────────────────────
df_customers = spark.read.format("delta").load(f"{silver_path}/customers")
df_customers = df_customers.drop("source_file", "pipeline")

# ── FIRST TIME LOAD CHECK ───────────────────────────
def table_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except:
        return False

if table_exists(f"{gold_path}/dim_customers"):
    print("Existing table — running SCD Type 2 MERGE")

    delta_table = DeltaTable.forPath(spark, f"{gold_path}/dim_customers")

    # Step 1 — Close changed records
    delta_table.alias("target").merge(
        df_customers.alias("source"),
        """target.customer_id = source.customer_id 
           AND target.is_current = true
           AND (target.country != source.country 
           OR target.city != source.city)"""
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "valid_to":   current_timestamp(),
        "updated_at": current_timestamp()
    }).execute()

    # Step 2 — Insert new/changed records
    df_new = df_customers \
        .withColumn("customer_key", monotonically_increasing_id()) \
        .withColumn("valid_from",   current_timestamp()) \
        .withColumn("valid_to",     lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",   lit(True))

    df_new.write.format("delta") \
               .mode("append") \
               .save(f"{gold_path}/dim_customers")

else:
    print("First time load — writing fresh")

    df_customers = df_customers \
        .withColumn("customer_key", monotonically_increasing_id()) \
        .withColumn("valid_from",   current_timestamp()) \
        .withColumn("valid_to",     lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",   lit(True))

    df_customers = df_customers.select(
        "customer_key", "customer_id", "company_name",
        "contact_name", "contact_title", "city", "country",
        "valid_from", "valid_to", "is_current",
        "created_at", "updated_at"
    )

    df_customers.write.format("delta") \
                      .mode("overwrite") \
                      .save(f"{gold_path}/dim_customers")

print("gold.dim_customers written successfully")

In [ ]:
# ── READ SILVER ─────────────────────────────────────
df_employees = spark.read.format("delta").load(f"{silver_path}/employees")
df_employees = df_employees.drop("source_file", "pipeline")


# ── FIRST TIME LOAD CHECK ───────────────────────────
def table_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except:
        return False

if table_exists(f"{gold_path}/dim_employees"):
    print("Existing table — running SCD Type 2 MERGE")

    delta_table = DeltaTable.forPath(spark, f"{gold_path}/dim_employees")

    # Step 1 — Close changed records
    delta_table.alias("target").merge(
        df_employees.alias("source"),
        """target.employee_id = source.employee_id
           AND target.is_current = true
           AND (target.title != source.title
           OR target.city != source.city)"""
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "valid_to":   current_timestamp(),
        "updated_at": current_timestamp()
    }).execute()

    # Step 2 — Insert new/changed records
    df_new = df_employees \
        .withColumn("employee_key", monotonically_increasing_id()) \
        .withColumn("valid_from",   current_timestamp()) \
        .withColumn("valid_to",     lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",   lit(True))

    df_new.write.format("delta") \
               .mode("append") \
               .save(f"{gold_path}/dim_employees")

else:
    print("First time load — writing fresh")

    df_employees = df_employees \
        .withColumn("employee_key", monotonically_increasing_id()) \
        .withColumn("valid_from",   current_timestamp()) \
        .withColumn("valid_to",     lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",   lit(True))

    df_employees = df_employees.select(
        "employee_key", "employee_id", "employee_name",
        "first_name", "last_name", "title",
        "city", "country", "reports_to",
        "valid_from", "valid_to", "is_current",
        "created_at", "updated_at"
    )

    df_employees.write.format("delta") \
                      .mode("overwrite") \
                      .save(f"{gold_path}/dim_employees")

print("gold.dim_employees written successfully")

In [ ]:
# ── READ SILVER ─────────────────────────────────────
df_products   = spark.read.format("delta").load(f"{silver_path}/products")
df_categories = spark.read.format("delta").load(f"{silver_path}/categories")

df_products   = df_products.drop("source_file", "pipeline")
df_categories = df_categories.drop("source_file", "pipeline")

# ── JOIN PRODUCTS + CATEGORIES ──────────────────────
df_products = df_products.join(
    df_categories.select("category_id", "category_name"),
    on="category_id",
    how="left"
)

# ── FIRST TIME LOAD CHECK ───────────────────────────
def table_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except:
        return False

if table_exists(f"{gold_path}/dim_products"):
    print("Existing table — running SCD Type 2 MERGE")

    delta_table = DeltaTable.forPath(spark, f"{gold_path}/dim_products")

    # Step 1 — Close changed records
    delta_table.alias("target").merge(
        df_products.alias("source"),
        """target.product_id = source.product_id
           AND target.is_current = true
           AND (target.unit_price != source.unit_price
           OR target.discontinued != source.discontinued)"""
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "valid_to":   current_timestamp(),
        "updated_at": current_timestamp()
    }).execute()

    # Step 2 — Insert new/changed records
    df_new = df_products \
        .withColumn("product_key", monotonically_increasing_id()) \
        .withColumn("valid_from",  current_timestamp()) \
        .withColumn("valid_to",    lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",  lit(True))

    df_new.write.format("delta") \
               .mode("append") \
               .save(f"{gold_path}/dim_products")

else:
    print("First time load — writing fresh")

    df_products = df_products \
        .withColumn("product_key", monotonically_increasing_id()) \
        .withColumn("valid_from",  current_timestamp()) \
        .withColumn("valid_to",    lit("9999-12-31").cast("timestamp")) \
        .withColumn("is_current",  lit(True))

    df_products = df_products.select(
        "product_key", "product_id", "product_name",
        "quantity_per_unit", "unit_price", "discontinued",
        "category_id", "category_name",
        "valid_from", "valid_to", "is_current",
        "created_at", "updated_at"
    )

    df_products.write.format("delta") \
                     .mode("overwrite") \
                     .save(f"{gold_path}/dim_products")

print("gold.dim_products written successfully")

In [ ]:
from pyspark.sql.types import IntegerType

# ── READ SILVER ─────────────────────────────────────
df_order_details = spark.read.format("delta").load(f"{silver_path}/order_details") \
                             .drop("created_at", "updated_at", "source_file", "pipeline")

df_orders = spark.read.format("delta").load(f"{silver_path}/orders") \
                      .drop("created_at", "updated_at", "source_file", "pipeline")

# ── READ GOLD DIMS FOR SURROGATE KEYS ───────────────
df_dim_customers = spark.read.format("delta").load(f"{gold_path}/dim_customers") \
                             .filter(col("is_current") == True) \
                             .select("customer_key", "customer_id")

df_dim_employees = spark.read.format("delta").load(f"{gold_path}/dim_employees") \
                             .filter(col("is_current") == True) \
                             .select("employee_key", "employee_id")

df_dim_products  = spark.read.format("delta").load(f"{gold_path}/dim_products") \
                             .filter(col("is_current") == True) \
                             .select("product_key", "product_id")

df_dim_shippers  = spark.read.format("delta").load(f"{gold_path}/dim_shippers") \
                             .select("shipper_key", "shipper_id")

df_dim_date      = spark.read.format("delta").load(f"{gold_path}/dim_date") \
                             .select("date_key", "full_date")

# ── JOIN order_details + orders ──────────────────────
df_fact = df_order_details.join(
    df_orders,
    on="order_id",
    how="left"
)

# date_key from order_date
df_fact = df_fact.withColumn("order_date_only", col("order_date").cast("date"))
df_dim_date = df_dim_date.withColumn("full_date", col("full_date").cast("date"))
df_fact = df_fact.join(
    df_dim_date.withColumnRenamed("full_date", "order_date_only"),
    on="order_date_only",
    how="left"
)

# customer_key
df_fact = df_fact.join(df_dim_customers, on="customer_id", how="left")

# employee_key
df_fact = df_fact.join(df_dim_employees, on="employee_id", how="left")

# product_key
df_fact = df_fact.join(df_dim_products, on="product_id", how="left")

# shipper_key
df_fact = df_fact.join(df_dim_shippers, on="shipper_id", how="left")

# ── ADD SURROGATE KEY ───────────────────────────────
df_fact = df_fact.withColumn("sales_key", monotonically_increasing_id())

# ── AUDIT COLUMNS ───────────────────────────────────
df_fact = df_fact \
    .withColumn("created_at", current_timestamp()) \
    .withColumn("updated_at", current_timestamp())

# ── SELECT FINAL COLUMNS ────────────────────────────
df_fact = df_fact.select(
    "sales_key",
    "order_id",
    "customer_key",
    "employee_key",
    "product_key",
    "shipper_key",
    "date_key",
    "unit_price",
    "quantity",
    "discount",
    "freight",
    "revenue",
    "created_at",
    "updated_at"
)

# ── WRITE GOLD ──────────────────────────────────────
df_fact.write.format("delta") \
             .mode("overwrite") \
             .save(f"{gold_path}/fact_sales")

print("gold.fact_sales written successfully")

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
import pandas as pd

# ── CONFIG ──────────────────────────────────────────
storage_account = "adlsadylearning"
container       = "ady-adf-practice"
silver_path     = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Silver"
gold_path       = f"abfss://{container}@{storage_account}.dfs.core.windows.net/NorthWind Project/Gold"


df_fact = spark.read.format("delta").load(f"{gold_path}/fact_sales")
df_fact.count()
